In [1]:
from mmml.interfaces.dcmInterface import build_mdcm_from_dcmnet, generate_dcm_xyz
fn = "/home/ericb/mmml_tutorial/cli/charmm_ml_comparison/charmm_ml_comparison.h5"

# From H5:
frames, charges_per_frame = build_mdcm_from_dcmnet(
    fn, average_over_frames=True, out_mdcm="meoh.mdcm"
)

In [2]:
# Python dcm.xyz without CHARMM:
import h5py
with h5py.File(fn, "r") as f:
    R = f["R"][0][:int(f["N"][0])]
generate_dcm_xyz(R, frames, charges_per_frame, "test_dcm.xyz")

In [3]:
import mmml
from mmml.interfaces.pycharmmInterface import import_pycharmm
import os
import sys
from pathlib import Path

from mmml.cli.make.make_res import main_loop
import argparse
import pycharmm


def main():
    args = argparse.Namespace(res="MEOH", skip_energy_show=True)
    print("=== 01: make_res programmatic ===")
    atoms = main_loop(args)
    print(f"Generated {len(atoms)} atoms")
    print("Output: pdb/initial.pdb, psf/initial.psf, xyz/initial.xyz, CHARMM topology files")


output =    main()


from pycharmm.coor import set_positions, get_positions


/home/ericb/mmml/mmml/data/charmm/top_all36_cgenff.rtf
/home/ericb/mmml/mmml/data/charmm/par_all36_cgenff.prm
CHARMM_HOME /home/ericb/mmml/setup/charmm
CHARMM_LIB_DIR /home/ericb/mmml/setup/charmm
  
 CHARMM>     BLOCK
 WARNING from DECODI -- Zero length string being converted to 0
 Block structure initialized with   3 blocks.
 All atoms have been assigned to block 1.
 All interaction coefficients have been set to unity.
  Setting number of block exclusions nblock_excldPairs=0
  
  BLOCK>            CALL 1 SELE ALL END
 SELRPN>      0 atoms have been selected out of      0
 The selected atoms have been reassigned to block   1
  
  BLOCK>              COEFF 1 1 1.0
  
  BLOCK>            END
 Matrix of Interaction Coefficients
 
    1.00000
    1.00000   1.00000
    1.00000   1.00000   1.00000
 Matrix of BOND Interaction Coefficients
 
    1.00000
    1.00000   1.00000
    1.00000   1.00000   1.00000
 Matrix of ANGLE Interaction Coefficients
 
    1.00000
    1.00000   1.00000
    1.000

In [4]:
pos = get_positions()

pos[:] = R

pos, R

set_positions(pos)

6

In [ ]:

dcm_script = """
open unit 11 card read name meoh.mdcm
!open unit 12 card read name water.kern
open unit 99 write card name dcm.xyz
DCM IUDCM 11 TSHIFT XYZ 99
!DCM KERN 12 IUDCM 11 TSHIFT XYZ 15
"""
pycharmm.lingo.charmm_script(dcm_script)
pycharmm.lingo.charmm_script("ENER")
pycharmm.lingo.charmm_script("STOP")


In [ ]:
import numpy as np

In [ ]:
np.array(pycharmm.coor.get_positions())

In [4]:
np.round(pycharmm.psf.get_amass())

array([12., 16.,  1.,  1.,  1.,  1.])

In [5]:
import ase
from ase.visualize import view

In [6]:
atoms = ase.io.read("pdb/initial.pdb")
Z = [_[0] for _ in pycharmm.psf.get_atype()]

In [7]:
atoms = ase.Atoms(Z, np.array(pycharmm.coor.get_positions()) )